# 11: End-to-End Automated 3D Object Isolation Pipeline

This notebook implements the complete pipeline from a raw Aria VRS recording to isolated 3D meshes of objects of interest.

### Pipeline stages:
1. **VRS Loading & Frame Extraction** — Decode the recording, extract RGB frames at 5 fps.
2. **Eye-Gaze Alignment** — Load MPS eye-gaze data, align temporally with RGB frames.
3. **Gaze Projection & Fixation Detection** — Project gaze to 2D, detect sustained fixations via I-DT.
4. **SAM2 Segmentation** — Generate masks using multi-run mono-prompt strategy (one SAM2 session per fixation).
5. **Image Preprocessing** — Crop distortion borders for Nerfstudio compatibility.
6. **Pose Estimation** — Run COLMAP via `ns-process-data`.
7. **NeRF Training** — Train a Nerfacto model on the scene.
8. **Geometry Export** — Export dense mesh from the trained model.
9. **Semantic Projection** — Project SAM2 masks onto mesh vertices using Nerfstudio pipeline cameras with majority voting.
10. **Subject Isolation** — Extract a clean sub-mesh containing only the voted subject vertices and faces.

As a sample, we will use the `table_objects.vrs` recordings which will be downsampled to 5 FPS.

## 11.1 Recording Configuration & Setup

All paths are derived from a single `RECORDING_NAME`.

In [1]:
import os, sys, glob, gc, math, time, subprocess
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image
from projectaria_tools.core import data_provider, mps
from projectaria_tools.core.sensor_data import TimeDomain

# Recording identifier
RECORDING_NAME = "table_objects"
PROJECT_ROOT_ABS = r"c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1"

# Input paths
DATA_ROOT = os.path.join("..", "data", "raw", RECORDING_NAME)
VRS_PATH  = os.path.join(DATA_ROOT, f"{RECORDING_NAME}.vrs")
MPS_ROOT  = os.path.join(DATA_ROOT, f"mps_{RECORDING_NAME}_vrs")
GAZE_DIR  = os.path.join(MPS_ROOT, "eye_gaze")
GAZE_CSV  = os.path.join(GAZE_DIR, "general_eye_gaze.csv")

# Segmentation output
SEG_ROOT   = os.path.join("..", "data", "outputs", "segmentation", RECORDING_NAME)
FRAMES_DIR = os.path.join(SEG_ROOT, "frames_5fps")
MASKS_DIR  = os.path.join(SEG_ROOT, "sam2", "masks")

# Nerfstudio pipeline
NS_ROOT       = os.path.join("..", "data", "outputs", "nerfstudio", RECORDING_NAME)
IMAGES_CROPPED = os.path.join(NS_ROOT, "images_cropped")
TRAINING_DIR   = os.path.join(NS_ROOT, "training")
EXPORT_DIR     = os.path.join(NS_ROOT, "exports")

for d in [FRAMES_DIR, MASKS_DIR, IMAGES_CROPPED, TRAINING_DIR, EXPORT_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Validate VRS
for label, path in {"VRS": VRS_PATH, "Gaze CSV": GAZE_CSV}.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} not found: {os.path.abspath(path)}")

print(f"Recording: {RECORDING_NAME}")
print(f"VRS: {os.path.abspath(VRS_PATH)}")
print(f"Frames: {os.path.abspath(FRAMES_DIR)}")
print(f"Masks: {os.path.abspath(MASKS_DIR)}")
print(f"NS root: {os.path.abspath(NS_ROOT)}")

Recording: table_objects
VRS: c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\raw\table_objects\table_objects.vrs
Frames: c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\segmentation\table_objects\frames_5fps
Masks: c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\segmentation\table_objects\sam2\masks
NS root: c:\Users\Gmalv\Desktop\Scuola_Universita\AriaGen1\data\outputs\nerfstudio\table_objects


### 11.1.1 VRS Access & Frame Extraction at 5 fps

Open the VRS, extract every 6th RGB frame (30 fps to 5 fps), and save as JPEG.

In [2]:
provider = data_provider.create_vrs_data_provider(VRS_PATH)
rgb_stream_id = provider.get_stream_id_from_label("camera-rgb")
rgb_config = provider.get_image_configuration(rgb_stream_id)
n_rgb_total = provider.get_num_data(rgb_stream_id)
IMG_W, IMG_H = rgb_config.image_width, rgb_config.image_height

print(f"RGB stream: {rgb_stream_id} | {n_rgb_total} frames | {IMG_W}x{IMG_H}")

RGB stream: 214-1 | 2764 frames | 1408x1408


In [3]:
def extract_rgb_frame(provider, stream_id, index):
    """Read one RGB frame from VRS by index, return as HxWx3 uint8 numpy array."""
    sample = provider.get_sensor_data_by_index(stream_id, index)
    for attr in ["image_data_and_record", "image_data", "image"]:
        if not hasattr(sample, attr):
            continue
        payload = getattr(sample, attr)
        payload = payload() if callable(payload) else payload
        if isinstance(payload, tuple):
            payload = payload[0]
        for arr_attr in ["to_numpy_array", "buffer"]:
            if hasattr(payload, arr_attr):
                arr = getattr(payload, arr_attr)
                arr = arr() if callable(arr) else arr
                arr = np.asarray(arr)
                if arr.ndim >= 2:
                    if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
                        arr = np.transpose(arr, (1, 2, 0))
                    return arr.astype(np.uint8)
    raise RuntimeError(f"Cannot extract image at index {index}")

# Validate
test = extract_rgb_frame(provider, rgb_stream_id, 0)
print(f"Extraction OK: {test.shape}")

Extraction OK: (1408, 1408, 3)


In [4]:
SUBSAMPLE_STRIDE = 6  # 30fps / 6 = 5fps
FPS = 5.0

indices_5fps = list(range(0, n_rgb_total, SUBSAMPLE_STRIDE))
n_frames_5fps = len(indices_5fps)
print(f"Subsampling: stride={SUBSAMPLE_STRIDE} -> {n_frames_5fps} frames at {FPS} fps")

existing = len([f for f in os.listdir(FRAMES_DIR) if f.endswith(".jpg")])
if existing >= n_frames_5fps:
    print(f"Already extracted ({existing} files), skipping.")
else:
    for out_idx, vrs_idx in enumerate(indices_5fps):
        frame = extract_rgb_frame(provider, rgb_stream_id, vrs_idx)
        Image.fromarray(frame).save(os.path.join(FRAMES_DIR, f"{out_idx:06d}.jpg"), quality=95)
        if (out_idx + 1) % 100 == 0:
            print(f"  [{out_idx+1}/{n_frames_5fps}]")
    print(f"[OK] {n_frames_5fps} frames saved.")

Subsampling: stride=6 -> 461 frames at 5.0 fps
Already extracted (461 files), skipping.


In [5]:
# Build 5fps timestamp array
rgb_timestamps_ns_30fps = np.array([
    provider.get_sensor_data_by_index(rgb_stream_id, i).get_time_ns(TimeDomain.DEVICE_TIME)
    for i in range(n_rgb_total)], dtype=np.int64)

rgb_timestamps_ns = rgb_timestamps_ns_30fps[indices_5fps]
rgb_duration_s = (rgb_timestamps_ns[-1] - rgb_timestamps_ns[0]) / 1e9
rgb_rate_hz = len(rgb_timestamps_ns) / rgb_duration_s
print(f"5fps frames: {len(rgb_timestamps_ns)} | Duration: {rgb_duration_s:.2f}s | Rate: {rgb_rate_hz:.2f} Hz")

5fps frames: 461 | Duration: 91.99s | Rate: 5.01 Hz
